# 1. Profiling

Primeira etapa do pipeline de ETL: entender schema, nulos e categorias dos
DataTables brutos exportados do jogo, antes de qualquer limpeza ou merge.

Este notebook só lê e inspeciona - não escreve nenhum arquivo.


In [1]:
import json
from pathlib import Path

import pandas as pd

# Notebook lives in notebooks/etl/, so the repo root is two levels up.
REPO_ROOT = Path("../..").resolve()

ROOT = REPO_ROOT / "data/Pal/Content"
PAL = ROOT / "Pal"
ITEM_DT = PAL / "DataTable/Item/DT_ItemDataTable_Common.json"
RECIPE_DT = PAL / "DataTable/Item/DT_ItemRecipeDataTable_Common.json"
BUILDOBJECT_DT = PAL / "DataTable/MapObject/Building/DT_BuildObjectDataTable_Common.json"
BENCH_RECIPES = REPO_ROOT / "src/data/bench_recipes.json"
NAMES_DT_EN = ROOT / "L10N/en/Pal/DataTable/Text/DT_ItemNameText_Common.json"
NAMES_DT_PT_BR = ROOT / "L10N/pt-BR/Pal/DataTable/Text/DT_ItemNameText_Common.json"


def load_rows(path):
    with open(path, encoding="utf-8") as f:
        return json.load(f)[0]["Rows"]


## Carregar as tabelas brutas em DataFrames


In [2]:
items_raw = load_rows(ITEM_DT)
recipes_raw = load_rows(RECIPE_DT)
buildings_raw = load_rows(BUILDOBJECT_DT)

items_df = pd.DataFrame.from_dict(items_raw, orient="index")
recipes_df = pd.DataFrame.from_dict(recipes_raw, orient="index")
buildings_df = pd.DataFrame.from_dict(buildings_raw, orient="index")

print(f"items: {items_df.shape}, recipes: {recipes_df.shape}, buildings: {buildings_df.shape}")


items: (2466, 53), recipes: (1414, 20), buildings: (498, 32)


## Schema: colunas e tipos

Antes de confiar em qualquer campo, olhar o que realmente existe em cada
tabela - DataTables da UE costumam ter campos que não são usados no jogo,
ou que só aparecem preenchidos em algumas rows.


In [3]:
items_df.dtypes


OverrideName                    str
OverrideDescription             str
IconName                        str
TypeA                           str
TypeB                           str
Rank                          int64
Rarity                        int64
MaxStackCount                 int64
Weight                      float64
Price                         int64
SortId                        int64
bInTreasureBox                 bool
bNotConsumed                   bool
bNotAvailableInPVP             bool
bEnableHandcraft               bool
bLegalInGame                   bool
TechnologyTreeLock            int64
ItemStaticClass                 str
ItemDynamicClass                str
ItemActorClass                  str
ItemStaticMeshName              str
VisualBlueprintClassName        str
VisualBlueprintClassSoft     object
DropItemType                    str
Editor_RowNameHash            int64
RestoreSatiety                int64
RestoreConcentration          int64
RestoreSanity               

In [4]:
recipes_df.dtypes


Product_Id                str
Product_Count           int64
WorkAmount            float64
WorkableAttribute       int64
UnlockItemID              str
Material1_Id              str
Material1_Count         int64
Material2_Id              str
Material2_Count         int64
Material3_Id              str
Material3_Count         int64
Material4_Id              str
Material4_Count         int64
Material5_Id              str
Material5_Count         int64
EnergyType                str
EnergyAmount            int64
CraftExpRate          float64
DenyRecipeChain        object
Editor_RowNameHash      int64
dtype: object

In [5]:
buildings_df.dtypes


MapObjectId                             str
TypeA                                   str
SortId                                int64
TypeB                                   str
TypeUIDisplay                           str
Rank                                  int64
BuildCapacity                         int64
RequiredBuildWorkAmount             float64
AssetValue                            int64
RequiredEnergyType                      str
ConsumeEnergySpeed                  float64
Material1_Id                            str
Material1_Count                       int64
Material2_Id                            str
Material2_Count                       int64
Material3_Id                            str
Material3_Count                       int64
Material4_Id                            str
Material4_Count                       int64
BlueprintItemID                         str
OverrideDescMsgID                       str
bInstallAtReticle                      bool
InstallNeighborThreshold        

## Nulos

Cuidado: a UE exporta ausência de valor como a **string literal `"None"`**,
não como null real do JSON. Um `.isna()` direto no DataFrame não pega isso -
por isso duas checagens separadas abaixo.


In [6]:
def none_string_rate(df):
    return (df == "None").sum().sort_values(ascending=False)

print("Colunas de items_df com valor sentinela 'None' (string, não nulo de verdade):")
none_string_rate(items_df).head(15)


Colunas de items_df com valor sentinela 'None' (string, não nulo de verdade):


ItemStaticMeshName          2450
PassiveSkillName4           2401
PassiveSkillName3           2346
PassiveSkillName2           2274
PassiveSkillName            1906
OverrideName                1889
OverrideDescription         1850
ItemActorClass              1650
ItemDynamicClass            1518
VisualBlueprintClassName    1273
ItemStaticClass             1252
GrantEffect2Time               0
GrantEffect3Id                 0
GrantEffect3Time               0
Durability                     0
dtype: int64

In [7]:
real_nulls = items_df.isna().sum()
real_nulls = real_nulls[real_nulls > 0]
print("Nulos reais (None do Python / NaN), se houver algum:")
real_nulls


Nulos reais (None do Python / NaN), se houver algum:


Series([], dtype: int64)

## Distribuição de categorias (`TypeA` / `TypeB` / `Rarity`)


In [8]:
items_df["TypeA"].value_counts()


TypeA
EPalItemTypeA::Blueprint              603
EPalItemTypeA::Weapon                 388
EPalItemTypeA::Armor                  355
EPalItemTypeA::Essential              334
EPalItemTypeA::Consume                237
EPalItemTypeA::Material               214
EPalItemTypeA::Accessory              145
EPalItemTypeA::Food                   124
EPalItemTypeA::Ammo                    38
EPalItemTypeA::SpecialWeapon           12
EPalItemTypeA::Glider                   8
EPalItemTypeA::CaptureItemModifier      7
EPalItemTypeA::MonsterEquipWeapon       1
Name: count, dtype: int64

In [9]:
items_df["TypeB"].value_counts()


TypeB
EPalItemTypeB::Blueprint                    604
EPalItemTypeB::ArmorHead                    202
EPalItemTypeB::Accessory                    145
EPalItemTypeB::ArmorBody                    145
EPalItemTypeB::Essential_PalGear            143
                                           ... 
EPalItemTypeB::MonsterEquipWeapon             1
EPalItemTypeB::ConsumePalLevelUp              1
EPalItemTypeB::ReturnToBaseCamp               1
EPalItemTypeB::ConsumePalRevive               1
EPalItemTypeB::ConsumeWorldTreeHolyWater      1
Name: count, Length: 70, dtype: int64

In [10]:
items_df["Rarity"].value_counts().sort_index()


Rarity
0     601
1     417
2     513
3     506
4     424
5       2
99      3
Name: count, dtype: int64

## Sinal de itens de debug/teste: `bLegalInGame`

Candidato a filtro de limpeza (etapa 2) - mas `bLegalInGame=False` sozinho
não é prova definitiva de "item de teste", só um sinal a investigar.


In [11]:
print(items_df["bLegalInGame"].value_counts())
suspects = items_df[items_df["bLegalInGame"] == False]
print(f"\n{len(suspects)} itens com bLegalInGame=False (candidatos a item de debug/teste):")
suspects[["TypeA", "TypeB", "Rarity"]].head(20)


bLegalInGame
True     1891
False     575
Name: count, dtype: int64

575 itens com bLegalInGame=False (candidatos a item de debug/teste):


,TypeA,TypeB,Rarity
AnimalSkin,EPalItemTypeA::Material,EPalItemTypeB::MaterialMonster,0
AnimalSkin2,EPalItemTypeA::Material,EPalItemTypeB::MaterialMonster,1
Scales,EPalItemTypeA::Material,EPalItemTypeB::MaterialMonster,0
Scales2,EPalItemTypeA::Material,EPalItemTypeB::MaterialMonster,1
Axe_Tier_03,EPalItemTypeA::Weapon,EPalItemTypeB::WeaponMelee,2
Bat_NPC,EPalItemTypeA::Weapon,EPalItemTypeB::WeaponMelee,0
Berries2,EPalItemTypeA::Food,EPalItemTypeB::FoodVegetable,1
CaptureRope,EPalItemTypeA::Weapon,EPalItemTypeB::SPWeaponCaptureRope,0
Claws,EPalItemTypeA::Material,EPalItemTypeB::MaterialMonster,0
Claws2,EPalItemTypeA::Material,EPalItemTypeB::MaterialMonster,1


## Casing inconsistente entre tabelas

Alguns `Material_Id`/`Product_Id` em `recipes_df` não batem exatamente com
o id na tabela de items (ex.: `"cloth"` em vez de `"Cloth"`). Detectar aqui
quantos casos existem antes de decidir a estratégia de resolução na limpeza.


In [12]:
item_ids = set(items_df.index)
item_ids_lower = {i.lower() for i in item_ids}

material_cols = [c for c in recipes_df.columns if c.startswith("Material") and c.endswith("_Id")]
referenced_ids = set(recipes_df["Product_Id"]) | set(recipes_df[material_cols].values.flatten())
referenced_ids -= {"None"}

case_mismatches = sorted(
    rid for rid in referenced_ids
    if rid not in item_ids and rid.lower() in item_ids_lower
)
print(f"{len(case_mismatches)} id(s) referenciados em recipes com casing diferente do item table:")
case_mismatches


6 id(s) referenciados em recipes com casing diferente do item table:


['FIber', 'Hotmilk', 'cloth', 'cloth2', 'stone', 'wood']

## Cobertura de nome por idioma (via chave padrão `ITEM_NAME_<id>`, ignorando `OverrideName`)


In [13]:
names_en = load_rows(NAMES_DT_EN)
names_pt = load_rows(NAMES_DT_PT_BR)

missing_en = [i for i in item_ids if f"ITEM_NAME_{i}" not in names_en]
missing_pt = [i for i in item_ids if f"ITEM_NAME_{i}" not in names_pt]
print(f"{len(missing_en)} items sem nome em EN pela chave padrão")
print(f"{len(missing_pt)} items sem nome em pt-BR pela chave padrão")
print("(um número alto aqui é esperado se muitos itens usam OverrideName - conferir amostra antes de tratar como bug)")


502 items sem nome em EN pela chave padrão
502 items sem nome em pt-BR pela chave padrão
(um número alto aqui é esperado se muitos itens usam OverrideName - conferir amostra antes de tratar como bug)
